<a id="mail-agent-title"></a>
# From RAG to Agent -- a Thunderbird Mail Assistant

**AI4Metascience School -- CNRS AISSAI Center**
*Domaine Saint-Paul, Saint-Remy-les-Chevreuse -- Sept/Oct 2026*

*(Continuation of `02_distributed_architecture_streamlit_litellm.ipynb` -- this is the
Thunderbird plugin teased at the end of that notebook.)*

---

This notebook does **not** build the plugin step by step the way notebooks 1 and 2 build their
pipelines. A browser extension can't run inside a Jupyter kernel, so the actual deliverable is a
standalone directory, **`thunderbird_agent/`**, at the repo root -- real files, loaded directly
into Thunderbird. This notebook is the pedagogical companion: what an *agent* is, in what way
this plugin qualifies as one, its architecture, why Thunderbird, and a guided walkthrough of the
code you'll actually load and run.

**What you will understand by the end**:

1. What "agent" means beyond "a chatbot with extra steps," and where this plugin sits on that
   spectrum.
2. Its architecture -- two swappable backends (the same local Ollama and remote LiteLLM proxy
   from notebooks 1 and 2), reused, not reinvented.
3. Why Thunderbird specifically, among the possible mail clients.
4. How each piece of `thunderbird_agent/` works, with the actual code.
5. How to load it and try it on a real email.
6. The scope decisions behind it, and concrete directions to take it further.

## Plan

- [1. What is an agent, and why is this plugin one?](#sec-1)
- [2. Architecture](#sec-2)
- [3. Why Thunderbird?](#sec-3)
- [4. Code walkthrough](#sec-4)
  - [4.1 The manifest: what the extension is allowed to touch](#sec-4-1)
  - [4.2 Perception: reading the displayed email](#sec-4-2)
    - [4.2.1 Attachments: text files read directly, PDFs read with a vendored parser](#sec-4-2-1)
    - [4.2.2 Showing it in the popup](#sec-4-2-2)
  - [4.3 Decision: calling the LLM, local or remote](#sec-4-3)
  - [4.4 Action: inserting into a real reply window](#sec-4-4)
  - [4.5 Memory: a small local writing-style history](#sec-4-5)
- [5. Install it and try it](#sec-5)
- [6. Scope decisions -- and why](#sec-6)
- [7. Going further](#sec-7)
- [Checkpoint](#checkpoint)

<a id="sec-1"></a>
## 1. What is an agent, and why is this plugin one?

A useful, non-hand-wavy definition: an **agent** is a system that (1) **perceives** something
about its environment, (2) uses an LLM to **decide** what to do about it, and (3) **acts** back
on that environment -- not just returns text to a human to act on themselves. The loop is
perceive &rarr; decide &rarr; act, and the "act" step is what separates an agent from a chatbot.

A plain chatbot (notebook 1's `rag_chat()`, or ChatGPT's web UI) perceives a typed question,
decides an answer, and "acts" by... printing text back at you. You are the one who reads it,
decides what to do, and does it (e.g. copy a paragraph into an email). Every action in the world
still routes through a human.

This plugin closes that loop one step further:

| Step | Chatbot (notebooks 1-2) | This plugin |
|---|---|---|
| **Perceive** | A question you typed | The email you're currently reading, read directly from Thunderbird's own message store -- no copy-paste |
| **Decide** | Retrieve + generate an answer | Generate a reply, optionally steered by your instructions |
| **Act** | Print the answer; you decide what to do with it | Open a **real** Thunderbird reply window with the draft already in the body -- a concrete state change in your actual mail client, not just text in a popup |

It is still a **narrow, human-in-the-loop** agent, deliberately: it never sends anything by
itself (see 6, "Scope decisions"). That constraint doesn't make it "not an agent" -- it makes it
a well-scoped one. The perceive/decide/act loop is what qualifies it; full autonomy is a separate
design axis, not a requirement for the term to apply. Compare it to the spectrum:

```text
Chatbot            -- perceive (your text) -> decide -> respond with text, human acts
This plugin         -- perceive (an email)  -> decide -> ACT (open a real, pre-filled reply)
Fully autonomous    -- perceive -> decide -> act -> perceive the result -> decide again -> ...
                        (e.g. an agent that also reads the reply it gets and follows up)
```

Notebook 1's confidence-banded generation (7.1) is itself a small piece of "deciding what to do,"
reused unchanged in spirit here: instead of a similarity threshold picking a sampling strategy,
here the user's own steering instructions (or their absence) shape the system prompt -- same
underlying idea, "let context change how the LLM is asked," applied to a different signal.

<a id="sec-2"></a>
## 2. Architecture

```mermaid
flowchart TB
    subgraph TB[Thunderbird]
        MSG[Currently displayed email<br/>+ its attachments]
        POPUP[Popup UI<br/>backend choice + steering prompt]
        BG[Background script<br/>background.js]
        COMPOSE[Native reply window<br/>signature + quoted text + Send]
    end

    MSG -- getFull + listAttachments<br/>+ getAttachmentFile --> BG
    BG -- text/PDF attachments only --> PDFJS[Vendored pdf.js<br/>PDF text extraction]
    POPUP <-- runtime.sendMessage --> BG
    BG -- fetch --> OLLAMA[Ollama<br/>localhost:11434<br/>notebook 1]
    BG -- fetch through SSH tunnel --> LITELLM[LiteLLM proxy<br/>127.0.0.1:4000<br/>notebook 2]
    BG -- compose.beginReply +<br/>setComposeDetails --> COMPOSE
    BG <--> STORAGE[(browser.storage.local<br/>settings + history)]
```

Two things worth noticing:

- **No new backend.** The "local" path is exactly notebook 1's Ollama; the "remote" path is
  exactly notebook 2's LiteLLM proxy, same URL, same model name, same participant key, same SSH
  tunnel. The extension is a new *client* for infrastructure you already built -- not a third
  deployment to maintain.
- **The background script is the only thing that talks to the network or to Thunderbird's
  compose API.** The popup only ever talks to the background script via
  `browser.runtime.sendMessage`. This isn't just tidiness: WebExtension popups are destroyed the
  instant they lose focus (click anywhere else and it's gone), so any state or in-flight request
  that lived *only* in the popup would vanish with it. The background page persists for the whole
  Thunderbird session, so it's the only place that can safely hold settings, history, and an
  in-flight LLM call.

<a id="sec-3"></a>
## 3. Why Thunderbird?

- **Free, open-source, cross-platform.** Windows, macOS, Linux -- the same extension works
  everywhere, no per-OS build. Matches this workshop's whole "free and open-source only" stance
  (README, Sections 10-11).
- **A real, documented extension API for composing** -- `messenger.compose.beginReply()` opens
  Thunderbird's own compose window, with the user's actual signature and quoting behavior. Many
  webmail providers (Gmail's web UI, for instance) don't expose anything close to this to a
  browser extension without going through a much more restrictive, cloud-tied add-on platform.
  Thunderbird's WebExtensions API is a close cousin of Firefox's -- same underlying model, mail-
  specific APIs layered on top (`messenger.messages`, `messenger.compose`,
  `messenger.messageDisplay`).
- **Already the mail client many French research labs standardize on** -- CNRS/CEA/university IT
  departments frequently deploy Thunderbird, which matters for "will this actually be usable at
  my lab" more than a client most participants don't run day to day.
- **No account, no cloud dependency.** Everything -- the extension, the local model, the remote
  proxy through your own SSH tunnel -- stays within infrastructure you control, consistent with
  every other notebook in this workshop.

<a id="sec-4"></a>
## 4. Code walkthrough

<a id="sec-4-1"></a>
### 4.1 The manifest: what the extension is allowed to touch

WebExtensions declare their capabilities upfront in `manifest.json` -- Thunderbird shows this
list to the user at install time, so it doubles as a trust boundary, not just config:

```json
{
  "manifest_version": 2,
  "name": "LaboBots Mail Agent",
  "version": "0.2.0",
  "description": "Draft email replies with a local (Ollama) or remote (LiteLLM proxy) LLM, inserted directly into Thunderbird's own reply window so your signature and Send button just work.",
  "author": "LaboBots workshop",

  "browser_specific_settings": {
    "gecko": {
      "id": "mail-agent@labobots.workshop",
      "strict_min_version": "115.0"
    }
  },

  "icons": {
    "32": "icons/icon.svg",
    "64": "icons/icon.svg",
    "128": "icons/icon.svg"
  },

  "permissions": [
    "messagesRead",
    "compose",
    "storage",
    "webRequest",
    "webRequestBlocking",
    "http://127.0.0.1/*",
    "http://localhost/*"
  ],

  "background": {
    "scripts": ["background.js"]
  },

  "message_display_action": {
    "default_title": "LaboBots Mail Agent",
    "default_icon": "icons/icon.svg",
    "default_popup": "popup/popup.html"
  },

  "options_ui": {
    "page": "options/options.html",
    "open_in_tab": true
  }
}
```

Notice what's *not* there: no `<all_urls>`, no permission to read arbitrary web pages, no access
to other extensions. `messagesRead` and `compose` are the two Thunderbird-specific permissions
that make this a *mail* agent rather than a generic one; `http://127.0.0.1/*` and
`http://localhost/*` are the only network hosts it can ever reach -- exactly the two backends from
notebooks 1 and 2, nothing else. `webRequest` + `webRequestBlocking` are there for a single, narrow job: rewriting
the `Origin` header of requests *to those same two hosts*, because Ollama rejects (HTTP 403) the
`moz-extension://` origin extensions send by default. Also absent on purpose: `compose.send` --
the extension opens and fills a reply, but has no permission to send it. `message_display_action` (rather than a permanent toolbar
`browser_action`) is deliberate too: the icon only appears -- and only makes sense to click --
when an email is actually open, mirroring the "perceive" step from Section 1.

<a id="sec-4-2"></a>
### 4.2 Perception: reading the displayed email

Thunderbird's `messages.getFull()` doesn't return a plain body string -- it returns a MIME part
tree (a message can have a `text/plain` part, a `text/html` part, attachments, nested
`multipart/*` wrappers...). `extractBodyFromPart` walks that tree looking for `text/plain` first,
falling back to `text/html` stripped of tags. Thunderbird 128+ offers
`messages.listInlineTextParts()`, which does this decoding for us, so it's tried first:

```javascript
function extractBodyFromPart(part, preferred = "text/plain") {
  if (!part) return null;
  if (part.contentType && part.contentType.startsWith(preferred) && part.body) {
    return part.body;
  }
  if (part.parts) {
    for (const child of part.parts) {
      const found = extractBodyFromPart(child, preferred);
      if (found) return found;
    }
  }
  return null;
}

function htmlToPlainText(html) {
  const doc = new DOMParser().parseFromString(html, "text/html");
  return doc.body ? doc.body.textContent.trim() : html;
}

async function getMessageBody(messageId) {
  // Thunderbird 128+ has a dedicated API that already decodes the inline text parts; older
  // versions fall back to walking the MIME tree ourselves.
  if (browser.messages.listInlineTextParts) {
    const parts = await browser.messages.listInlineTextParts(messageId);
    const plain = parts.find((p) => p.contentType === "text/plain");
    if (plain && plain.content.trim()) return plain.content;
    const html = parts.find((p) => p.contentType === "text/html");
    if (html) return htmlToPlainText(html.content);
  }
  const full = await browser.messages.getFull(messageId);
  const plain = extractBodyFromPart(full, "text/plain");
  if (plain) return plain;
  const html = extractBodyFromPart(full, "text/html");
  return html ? htmlToPlainText(html) : "(could not extract a readable body)";
}

async function getDisplayedEmail(tabId) {
  // The popup passes the id of the tab it was opened from: the background page has no "current
  // tab" of its own, and getDisplayedMessage() needs to know which tab's message to read.
  const message = await browser.messageDisplay.getDisplayedMessage(tabId);
  if (!message) {
    throw new Error("No single message is currently displayed. Open (or select) one email first.");
  }
  const body = await getMessageBody(message.id);
  const attachments = await getAttachmentsWithText(message.id); // see 4.2.1 below
  return {
    messageId: message.id,
    subject: message.subject,
    from: message.author,
    body: body.slice(0, 6000), // keep the prompt a reasonable size for a small local model
    attachments,
  };
}
```

Note the `tabId` parameter: the background page has no "current tab" of its own, so the popup
looks up the tab it was opened from (`browser.tabs.query({ active: true, currentWindow: true })`)
and passes its id along -- `getDisplayedMessage()` needs to know *which* tab's message to read.

The fallback walker is a **simplified** extractor, called out as a known limitation in the README -- real-world
MIME structures (inline images, `multipart/related`, unusual encodings) can defeat a hand-rolled
walker. Good enough for the workshop's typical plain-text/HTML lab emails; a production add-on
would reach for a proper MIME-parsing library instead.

<a id="sec-4-2-1"></a>
### 4.2.1 Attachments: text files read directly, PDFs read with a vendored parser

`getDisplayedEmail` (above) now also calls `getAttachmentsWithText()`, which reads every
attachment Thunderbird reports for the message and decides, per attachment, whether it can
actually contribute to the prompt:

```javascript
function isTextAttachment(name, contentType) {
  if (contentType && contentType.startsWith("text/")) return true;
  if (contentType === "application/json") return true;
  const lower = (name || "").toLowerCase();
  return TEXT_ATTACHMENT_EXTENSIONS.some((ext) => lower.endsWith(ext));
}

function isPdfAttachment(name, contentType) {
  if (contentType === "application/pdf") return true;
  return (name || "").toLowerCase().endsWith(".pdf");
}

async function getAttachmentsWithText(messageId) {
  const list = await browser.messages.listAttachments(messageId);
  const results = [];
  for (const att of list) {
    const info = { name: att.name, contentType: att.contentType || "", textIncluded: false, text: "", note: "" };
    try {
      if (isTextAttachment(att.name, att.contentType)) {
        const file = await browser.messages.getAttachmentFile(messageId, att.partName);
        info.text = (await file.text()).slice(0, ATTACHMENT_TEXT_MAX_CHARS);
        info.textIncluded = info.text.trim().length > 0;
      } else if (isPdfAttachment(att.name, att.contentType)) {
        const file = await browser.messages.getAttachmentFile(messageId, att.partName);
        const bytes = new Uint8Array(await file.arrayBuffer());
        info.text = (await extractPdfText(bytes)).slice(0, ATTACHMENT_TEXT_MAX_CHARS);
        info.textIncluded = info.text.trim().length > 0;
        if (!info.textIncluded) info.note = "no extractable text (likely a scanned/image-only PDF)";
      } else {
        info.note = "not a text file or PDF -- not read";
      }
    } catch (err) {
      info.note = `could not read this attachment (${err.message})`;
    }
    results.push(info);
  }
  return results;
}
```

Three design choices worth calling out:

- **Never throws for one bad attachment.** Each attachment is wrapped in its own `try`/`catch` --
  a corrupt PDF or an unreadable file degrades to "not read" for that one attachment, instead of
  failing the whole draft.
- **PDFs are parsed with a vendored copy of [pdf.js](../thunderbird_agent/vendor/pdfjs/README.md)**
  (Mozilla's own PDF engine, Apache-2.0), loaded lazily via a dynamic `import()` of a local
  `.mjs` file -- no new manifest permission needed (`messagesRead` already covers attachments,
  and the extension never fetches pdf.js from a CDN, keeping the "fully offline, only talks to
  localhost" property from Section 3):
  ```javascript
  let pdfjsLibPromise = null;
  function loadPdfJs() {
    if (!pdfjsLibPromise) {
      pdfjsLibPromise = import(browser.runtime.getURL("vendor/pdfjs/pdf.min.mjs")).then((lib) => {
        lib.GlobalWorkerOptions.workerSrc = browser.runtime.getURL("vendor/pdfjs/pdf.worker.min.mjs");
        return lib;
      });
    }
    return pdfjsLibPromise;
  }
  ```
- **Everything else is only *noticed*, never guessed at.** An image, a `.docx`, a scanned PDF with
  no text layer -- these get a `note` explaining why they weren't read, shown in the popup (4.2.2)
  and never silently fabricated into the prompt.

Extracted text (capped per attachment, see `ATTACHMENT_TEXT_MAX_CHARS`) is folded into the same
user prompt as the email body in `generateDraft` (4.3), clearly delimited per attachment so the
model can tell "email body" from "attachment N" apart.

<a id="sec-4-2-2"></a>
### 4.2.2 Showing it in the popup

Before you even click "Generate draft", the popup lists what it found -- this is the moment a
demo audience actually *sees* the extra capability, not just hears about it:

```javascript
function attachmentsSummaryHtml(attachments) {
  if (!attachments || attachments.length === 0) return "";
  const items = attachments
    .map((a) => {
      const icon = a.textIncluded ? "📄" : a.contentType.startsWith("image/") ? "🖼️" : "📎";
      const status = a.textIncluded ? "content included" : a.note || "not read";
      return `<li>${icon} ${escapeHtml(a.name)} <span class="muted">(${escapeHtml(status)})</span></li>`;
    })
    .join("");
  return `<ul class="attachments">${items}</ul>`;
}
```

<p align="center">
  <img src="assets/screenshots/thunderbird_agent_popup_attachment.png" alt="LaboBots Mail Agent popup showing a PDF attachment with content included" width="480"><br>
  <sub><i>The popup on a real email with a PDF attachment -- "AI_Project_Followup_Brief.pdf
  (content included)" is exactly <code>attachmentsSummaryHtml()</code> rendering the 📄 case from
  <code>getAttachmentsWithText()</code> (4.2.1): the PDF was parsed, and its text is about to be
  folded into the prompt below.</i></sub>
</p>


<a id="sec-4-3"></a>
### 4.3 Decision: calling the LLM, local or remote

Two thin wrappers, one per backend -- notice they mirror notebook 1's `ollama.chat()` call and
notebook 2's `call_remote_llm()` almost line for line, just in JavaScript's `fetch` instead of
Python's `requests`:

```javascript
async function callOllama(settings, messages) {
  const resp = await fetch(`${settings.ollama_url}/api/chat`, {
    method: "POST",
    headers: { "Content-Type": "application/json" },
    body: JSON.stringify({ model: settings.ollama_model, messages, stream: false }),
  });
  if (!resp.ok) {
    throw new Error(`Ollama returned HTTP ${resp.status}. Is 'ollama serve' running locally?`);
  }
  const data = await resp.json();
  return data.message.content;
}

async function callLiteLLM(settings, messages) {
  if (!settings.litellm_key) {
    throw new Error("No LiteLLM key configured -- set it in this extension's Options page.");
  }
  const resp = await fetch(`${settings.litellm_url}/v1/chat/completions`, {
    method: "POST",
    headers: {
      "Content-Type": "application/json",
      Authorization: `Bearer ${settings.litellm_key}`,
    },
    body: JSON.stringify({ model: settings.litellm_model, messages }),
  });
  if (!resp.ok) {
    const text = await resp.text().catch(() => "");
    throw new Error(`LiteLLM proxy returned HTTP ${resp.status}. ${text} -- is the SSH tunnel to the remote host open (manage_remote_rag.sh tunnel)?`);
  }
  const data = await resp.json();
  return data.choices[0].message.content;
}

async function generateDraft({ email, steeringPrompt, backend }) {
  const settings = await getSettings();
  const historyContext = buildHistoryContext(settings.history);

  const userPrompt = `Original email
From: ${email.from}
Subject: ${email.subject}

${email.body}

---
${steeringPrompt ? `Steering instructions from the user: ${steeringPrompt}` : "No specific steering instructions -- use your best judgment for a reasonable reply."}${historyContext}

Draft the reply now.`;

  const messages = [
    { role: "system", content: DRAFT_SYSTEM_PROMPT },
    { role: "user", content: userPrompt },
  ];

  const draft = backend === "remote"
    ? await callLiteLLM(settings, messages)
    : await callOllama(settings, messages);

  return draft.trim();
}
```

The system prompt (`DRAFT_SYSTEM_PROMPT`, defined earlier in the file) is where the "decide" step
actually lives: it tells the model to write *only* the reply body (no greeting, no sign-off --
Thunderbird's own signature insertion handles that), to match the original email's language, and
to treat any steering instructions as authoritative.

<a id="sec-4-4"></a>
### 4.4 Action: inserting into a real reply window

This is the step that makes it an agent rather than a chatbot (Section 1) -- a real, observable
change in Thunderbird's own state, not text handed back to the user to act on themselves:

```javascript
function escapeHtml(s) {
  return s.replace(/&/g, "&amp;").replace(/</g, "&lt;").replace(/>/g, "&gt;");
}

function draftToHtml(draftText) {
  return draftText
    .split(/\n\s*\n/)
    .map((para) => `<p>${escapeHtml(para).replace(/\n/g, "<br>")}</p>`)
    .join("");
}

async function insertIntoReply({ messageId, draftText }) {
  const composeTab = await browser.compose.beginReply(messageId, "replyToSender");
  // setComposeDetails({body}) REPLACES the whole body, so we first read what Thunderbird put
  // there (quoted original + the user's signature) and prepend our draft to it. The editor can
  // still be empty for a moment right after beginReply resolves, hence the short retry loop.
  let details = await browser.compose.getComposeDetails(composeTab.id);
  for (let i = 0; i < 10 && !(details.isPlainText ? details.plainTextBody : details.body); i++) {
    await new Promise((r) => setTimeout(r, 200));
    details = await browser.compose.getComposeDetails(composeTab.id);
  }

  if (details.isPlainText) {
    await browser.compose.setComposeDetails(composeTab.id, {
      plainTextBody: `${draftText}\n\n${details.plainTextBody || ""}`,
    });
  } else {
    const existing = details.body || "<html><body></body></html>";
    const draftHtml = draftToHtml(draftText);
    const body = /<body[^>]*>/i.test(existing)
      ? existing.replace(/<body[^>]*>/i, (tag) => `${tag}${draftHtml}<br>`)
      : draftHtml + existing;
    await browser.compose.setComposeDetails(composeTab.id, { body });
  }
  return composeTab.id;
}
```

The comment explains the one non-obvious decision: `setComposeDetails({ body })` does **not**
prepend -- it *replaces* the whole message body. By the time `beginReply` returns, Thunderbird
has already built the quoted original text and inserted the user's signature, so we first read
that content back with `getComposeDetails` and write back *draft + existing content*. Both
compose modes are handled: plain-text (`plainTextBody`) and HTML (`body`, with the draft inserted
right after the `<body>` tag). Skip the read-back and you'd silently lose the signature and the
quote -- exactly the failure mode the user asked this plugin to avoid.

<a id="sec-4-5"></a>
### 4.5 Memory: a small local writing-style history

```javascript
function buildHistoryContext(history) {
  const recent = history.slice(-MAX_HISTORY_IN_PROMPT);
  if (recent.length === 0) return "";
  const examples = recent
    .map((h, i) => `Previous reply example ${i + 1} (style reference only):\n${h.draft}`)
    .join("\n\n");
  return `\n\nHere are a few of this user's own previous replies, for style reference only:\n\n${examples}`;
}

async function recordHistory({ subject, steeringPrompt, draft }) {
  const settings = await getSettings();
  const history = [...settings.history, { timestamp: Date.now(), subject, steeringPrompt, draft }];
  await saveSetting({ history: history.slice(-MAX_HISTORY) });
}
```

Every accepted draft is appended to a capped local list (`MAX_HISTORY = 20`); the next
generation's prompt folds the last few of those in as style examples ("here's how this person
usually writes"), the same way notebook 1's Section 7.3 "small-to-big" expansion folds extra
context into a prompt without changing the retrieval logic itself. See Section 6 for why this
stays deliberately simple rather than reaching for real fine-tuning or style embeddings.

<a id="sec-5"></a>
## 5. Install it and try it

Nothing in this section runs from the notebook -- a browser extension can't execute inside a
Jupyter kernel. Follow along in Thunderbird itself; full details are in
`thunderbird_agent/README.md`.

1. **Prerequisite**: notebook 1's Ollama setup (`llama3.2:3b` pulled and running), and/or
   notebook 2's SSH tunnel open (`./rag_workshop/manage_remote_rag.sh tunnel`) if you want to try
   the remote backend.
2. Build the installable package with `./thunderbird_agent/build.sh` (creates
   `thunderbird_agent/dist/labobots-mail-agent-<version>.xpi`), then in Thunderbird:
   **Tools -> Add-ons and Themes** -> gear icon -> **Install Add-on From File...** -> pick that
   `.xpi`.
3. Open the extension's **Options** page and fill in the remote backend's participant key (the
   local backend's defaults should already match notebook 1).
4. Open any email, click the LaboBots Mail Agent icon, pick a backend, optionally add steering
   instructions, **Generate draft**, review/edit it, **Insert into reply**.
5. Check: does the opened reply window have your real signature? Is the original message quoted
   below, as Thunderbird normally does? Both should be untouched by the extension (Section 4.4).

**Permanent vs temporary**: Thunderbird doesn't require add-ons to be signed, so the `.xpi`
above stays installed across restarts. While iterating on the code, the quicker loop is
**Tools -> Developer Tools -> Debug Add-ons** -> **Load Temporary Add-on...** ->
`thunderbird_agent/manifest.json`, then **Reload** after each edit (unloaded on restart).

<p align="center">
  <img src="assets/screenshots/thunderbird_agent_reply_draft.png" alt="A real Thunderbird reply window with a draft generated from the email and its PDF attachment" width="760"><br>
  <sub><i>The result: a real reply window (<code>compose.beginReply</code>, Section 4.4), signature
  and quoted original untouched, with a generated draft that references "pilot validation code
  (ATLAS-47)" and "action A2 / Check PDF text extraction" -- details that only exist in the
  attached PDF, not in the email body itself. This is the attachment-reading feature (4.2.1)
  actually influencing the draft, not just being detected in the popup.</i></sub>
</p>

<a id="sec-6"></a>
## 6. Scope decisions -- and why

Two choices were made deliberately narrow for this workshop, both easy to widen later without
restructuring what already exists:

**No retrieval (RAG) grounding in the MVP.** The draft is generated from the email content plus
your steering instructions -- it does not consult the CC/lab documentation collection from
notebook 1 the way `rag_chat()` does. Adding that would mean a WebExtension somehow running BGE-M3
encoding and querying Chroma, neither of which a browser extension can do natively -- it would
need a small local Python HTTP server (a "sidecar") exposing `hybrid_retrieve()` over `fetch`,
which `background.js` would call before building the prompt. That's a real, buildable extension
(Section 7) -- just more moving parts to stand up and keep running during a one-week-away
workshop than a first working version needs.

**A flat local history, not real fine-tuning or style embeddings.** "Reused by the model over
time" could mean anything from a rolling few-shot list (what's implemented) to actually
fine-tuning a personal adapter per user, or embedding past emails into their own small vector
index. Both of those are legitimate, more powerful designs -- and both need a training or
indexing pipeline that doesn't exist yet anywhere in this workshop. The flat history captures the
same *intent* ("the model should sound like this person over time") with zero new
infrastructure: it's already-working code (`browser.storage.local`), not a new subsystem.

Neither choice is permanent -- they're the smallest version that still does the thing the project
description asked for (a working agent, a real send button, a persistent profile), leaving the
bigger versions as well-scoped next steps instead of blockers to a first working demo.

<a id="sec-7"></a>
## 7. Going further

- **RAG-grounded drafts**: a small `rag_workshop/agent_sidecar.py` (FastAPI/Flask) wrapping
  notebook 1's `hybrid_retrieve()` + `rag_chat()` behind a `/draft` HTTP endpoint that
  `background.js` calls instead of (or in addition to) the raw LLM -- reuses the exact retrieval
  code from Sections 5-7, no new retrieval logic to write.
- **Streaming**: both Ollama and LiteLLM support streamed responses
  (`stream: true` / server-sent events); the popup could show tokens arriving instead of a single
  blocking wait, meaningfully better UX for a slow local model.
- **Real style memory**: embed each accepted draft (BGE-M3, same model as notebook 1) instead of
  storing raw text, and retrieve the *most similar* past drafts as few-shot context instead of
  just "the last N" -- turns Section 4.5's flat list into a small personal RAG index.
- **Multi-account / multi-identity signatures**: `messenger.identities` can enumerate a user's
  configured identities if they run more than one email account through Thunderbird; `beginReply`
  already picks the right one automatically for a given message, but a "which identity" selector
  in Options would matter for anyone with a second (e.g. personal) account configured.
- **Distribution**: publishing on addons.thunderbird.net (which signs the package and handles
  automatic updates) instead of handing labmates the `.xpi` from `build.sh`.

<a id="checkpoint"></a>
## Checkpoint

At this point you have:

- ✅ A working definition of "agent" (perceive -> decide -> act) and a concrete example of where
  a RAG chatbot stops and an agent begins.
- ✅ A real Thunderbird extension (`thunderbird_agent/`) reusing notebooks 1 and 2's exact LLM
  backends, no new infrastructure to deploy.
- ✅ An understanding of every moving part: manifest permissions, MIME body extraction,
  text/PDF attachment reading (via a vendored pdf.js), the two LLM callers, the compose-insertion
  ordering trick, and the local history mechanism.
- ✅ It loaded and tested in your own Thunderbird, drafting and inserting a real reply.
- ✅ A clear map of what was deliberately left out (RAG grounding, real fine-tuning) and how to
  add either one later without starting over.

This closes the loop this whole workshop has been building toward: from a from-scratch RAG
pipeline on one laptop (notebook 1), to a distributed, multi-user, authenticated deployment
(notebook 2), to an agent that acts on your own tools, not just answers questions about them.

In [1]:
import json
import shutil
import subprocess
from pathlib import Path

AGENT_DIR = Path("thunderbird_agent")

manifest = json.loads((AGENT_DIR / "manifest.json").read_text())
print("manifest.json: valid JSON, name =", manifest["name"])

referenced_paths = [
    manifest["background"]["scripts"][0],
    manifest["message_display_action"]["default_popup"],
    manifest["options_ui"]["page"],
] + list(manifest["icons"].values())

missing = [p for p in referenced_paths if not (AGENT_DIR / p).exists()]
print("Referenced files missing:", missing if missing else "none")

js_files = sorted(AGENT_DIR.rglob("*.js"))
if shutil.which("node") is None:
    print("node not installed -- skipping JS syntax check")
for js_file in js_files if shutil.which("node") else []:
    result = subprocess.run(["node", "--check", str(js_file)], capture_output=True, text=True)
    status = "OK" if result.returncode == 0 else f"SYNTAX ERROR: {result.stderr.strip()}"
    print(f"{js_file}: {status}")

html_files = sorted(AGENT_DIR.rglob("*.html"))
print(f"\nFound {len(js_files)} JS files and {len(html_files)} HTML files under {AGENT_DIR}/.")

manifest.json: valid JSON, name = LaboBots Mail Agent
Referenced files missing: none
thunderbird_agent/background.js: OK


thunderbird_agent/options/options.js: OK
thunderbird_agent/popup/popup.js: OK

Found 3 JS files and 2 HTML files under thunderbird_agent/.
